## 1. Prepare the stance-detection post IDs

The `user_post_list_for_stance_detection.parquet` file contains one row per user.
For each user, `PostIdList` contains the IDs of the posts that are part of their
posting history for stance detection.

Our goal in this step is to transform:

| UserId | PostIdList |
|---|---|
| 9 | [12167384, 4538012, 7809, ...] |

into:

| UserId | PostId |
|---|---|
| 9 | 12167384 |
| 9 | 4538012 |
| 9 | 7809 |

This exploded table can then be used to select only the required posts from the
much larger `user_post_history_dataset.parquet`.

In [28]:
import os
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.compute as pc


# Load the list of posts used for stance detection
# Each row represents one user and contains a list of PostIds
post_lists = pq.read_table(
    "../data/raw/user_post_list_for_stance_detection.parquet"
)

print(f"Users in original table: {post_lists.num_rows:,}")
display(post_lists.slice(0, 5).to_pandas())
print(post_lists.schema)
print(post_lists.slice(0, 5))

Users in original table: 8,467


,UserId,PostIdList
0,9,"[12167384, 4538012, 7809, 3924058, 7807, 48816..."
1,10,"[2620214, 12666, 12665, 12674, 12248774, 16645..."
2,49,"[13503, 13501, 13496, 13495, 13493, 13491, 134..."
3,105,"[14930, 14927, 14924, 14921, 369225, 14979, 14..."
4,144,"[15305, 15303, 15301, 15300, 15298, 15297, 152..."


UserId: int64
PostIdList: list<element: int64>
  child 0, element: int64
-- schema metadata --
pandas: '{"index_columns": [{"kind": "range", "name": null, "start": 0, "' + 495
pyarrow.Table
UserId: int64
PostIdList: list<element: int64>
  child 0, element: int64
----
UserId: [[9,10,49,105,144]]
PostIdList: [[[12167384,4538012,7809,3924058,7807,...,3991,3952,3946,9737956,12168218],[2620214,12666,12665,12674,12248774,...,11784,11783,11781,16209872,4915960],[13503,13501,13496,13495,13493,...,13416,13415,13410,13405,894684],[14930,14927,14924,14921,369225,...,14764,3136597,14763,14762,14761],[15305,15303,15301,15300,15298,...,15105,15104,15103,15101,15100]]]


In [29]:
# Flatten all PostId lists into one long PostId column
post_ids = pc.list_flatten(post_lists["PostIdList"])

# Get the index of the original user row for every flattened PostId
parent_indices = pc.list_parent_indices(post_lists["PostIdList"])

# Repeat each UserId for all PostIds belonging to that user
user_ids = pc.take(
    post_lists["UserId"],
    parent_indices
)

# Create the flat UserId–PostId table
exploded_posts = pa.table({
    "UserId": user_ids,
    "PostId": post_ids
})

display(exploded_posts.slice(0, 10).to_pandas())

,UserId,PostId
0,9,12167384
1,9,4538012
2,9,7809
3,9,3924058
4,9,7807
5,9,4881667
6,9,7805
7,9,7793
8,9,5778
9,9,5775


In [9]:
# Sanity checks to make sure the transformation worked as expected
print(f"User-post pairs: {exploded_posts.num_rows:,}")
print(
    f"Unique users: "
    f"{pc.count_distinct(exploded_posts['UserId']).as_py():,}"
)
print(
    f"Unique posts: "
    f"{pc.count_distinct(exploded_posts['PostId']).as_py():,}"
)

# Save the result 
output_path = "../data/preprocessed/stance_detection_post_ids.parquet"

pq.write_table(
    exploded_posts,
    output_path,
    compression="snappy"
)

print(f"\nSaved to: {output_path}")

User-post pairs: 2,845,217
Unique users: 8,467
Unique posts: 1,768,725

Saved to: ../data/preprocessed/stance_detection_post_ids.parquet


In [ ]:
# Further sanity checks
# 1. Check whether the exact same UserId–PostId pair appears more than once
df_check = exploded_posts.to_pandas()

print(
    "Duplicate user-post pairs:",
    df_check.duplicated(["UserId", "PostId"]).sum()
)

# 2. Inspect how many posts each user has
posts_per_user = df_check.groupby("UserId").size()

print(posts_per_user.describe())

Unique users: 8467
Duplicate user-post pairs: 0
count    8467.000000
mean      336.036022
std       351.754112
min         1.000000
25%        69.000000
50%       166.000000
75%       519.500000
max      1000.000000
dtype: float64


---
## 2. Extract the relevant posts from the full posting history

The full `user_post_history_dataset.parquet` contains more than 18 million posts.
However, only the posts listed in `stance_detection_post_ids.parquet` are relevant
for our stance-detection experiments.

We therefore:

1. Load the previously created `(UserId, PostId)` pairs.
2. Read the large posting-history file in batches.
3. Keep only rows whose `UserId` and `PostId` occur in our stance-detection list.
4. Retain the columns required for the later experiments:
   - `UserId`
   - `PostId`
   - `PostTime`
   - `Content`
5. Write the matching posts directly to a new Parquet file.

This avoids loading the complete 18-million-row dataset into memory.

In [ ]:
# Note: as the file "user_post_history_dataset.parquet" is relatively large, we cannot upload the original 2.1 GB file to github
# therefore this code will only work with the user_post_histpry_dataset.parquet uploaded to the data/raw folder.
# We've run the code locally and uploaded the preprocessed file to data/preprocessed/filtered_post_history 

# Paths
post_ids_path = "../data/preprocessed/stance_detection_post_ids.parquet"
history_path = "../data/raw/user_post_history_dataset.parquet"

output_path = "../data/preprocessed/filtered_post_history.parquet"

# Make sure the output directory exists
os.makedirs(os.path.dirname(output_path), exist_ok=True)

# Load the UserId–PostId pairs that we want to keep.
# This table is small enough to keep in memory.
wanted_posts = pq.read_table(post_ids_path)

# Open the large posting-history file without loading it completely.
history_file = pq.ParquetFile(history_path)

print(f"Wanted user-post pairs: {wanted_posts.num_rows:,}")
print(f"Rows in full post history: {history_file.metadata.num_rows:,}")
print(f"Row groups: {history_file.num_row_groups}")

FileNotFoundError: [WinError 2] Failed to open local file '../data/raw/user_post_history_dataset.parquet'. Detail: [Windows error 2] Das System kann die angegebene Datei nicht finden.


In [ ]:
# We only retain columns that are relevant for our later experiments
columns_to_keep = [
    "UserId",
    "PostId",
    "PostTime",
    "Content"
]

writer = None
matched_rows = 0

try:
    # Read the 18M-row history incrementally instead of loading it all at once
    for batch_number, batch in enumerate(
        history_file.iter_batches(
            batch_size=250_000,
            columns=columns_to_keep,
            use_threads=True
        ),
        start=1
    ):
        batch_table = pa.Table.from_batches([batch])

        # Inner join keeps only rows whose UserId AND PostId occur in our stance-detection post list
        matched = batch_table.join(
            wanted_posts,
            keys=["UserId", "PostId"],
            join_type="inner"
        )

        if matched.num_rows == 0:
            continue

        # Keep a predictable column order
        matched = matched.select(columns_to_keep)

        # Create the output writer after seeing the schema of the first result
        if writer is None:
            writer = pq.ParquetWriter(
                output_path,
                matched.schema,
                compression="snappy"
            )

        # Write immediately instead of collecting all results in RAM
        writer.write_table(matched)

        matched_rows += matched.num_rows

        print(
            f"Batch {batch_number}: "
            f"{matched.num_rows:,} matches "
            f"({matched_rows:,} total)"
        )

finally:
    if writer is not None:
        writer.close()

print(f"\nFinished.")
print(f"Matched posts: {matched_rows:,}")
print(f"Saved to: {output_path}")

In [30]:
output_path = "../data/preprocessed/filtered_post_history.parquet"
filtered_history = pq.read_table(output_path)

print(f"Rows: {filtered_history.num_rows:,}")
print(f"Columns: {filtered_history.num_columns}")
print()

print("Schema:")
print(filtered_history.schema)

print("\nFirst rows:")
display(filtered_history.slice(0, 10).to_pandas())

Rows: 2,845,217
Columns: 4

Schema:
UserId: int64
PostId: int64
PostTime: string
Content: string

First rows:


,UserId,PostId,PostTime,Content
0,181404,8420847,2024-11-19T05:05:01.485Z,My head hurts thinking about how illogical thi...
1,78674,3651245,2024-11-19T05:04:59.651Z,Howard Lutnick and Scott Bessent’s battle to b...
2,255778,11675952,2024-11-19T05:04:59.247Z,"Geordi: ""I always thought I could work with an..."
3,350049,16002791,2024-11-19T05:04:52.964Z,School vouchers have always been about allowin...
4,251056,11357083,2024-11-19T05:04:52.817Z,You just know trolls will get off on being ins...
5,197218,13305056,2024-11-19T06:46:30.421Z,Late dinner post 🍽️🍜\n(🌱 homegrown)\nChowder c...
6,22500,1132145,2024-11-19T05:04:48.611Z,One of the reasons Trump wants to do this mass...
7,85586,3996773,2024-11-19T05:04:46.811Z,I’m not going to reveal what I got my niece an...
8,247291,11245294,2024-11-19T05:04:46.319Z,
9,58969,2647036,2024-11-19T05:04:45.797Z,In answer to a growing surge for violence on t...


In [ ]:
# Sanity checks
expected = wanted_posts.num_rows
actual = filtered_history.num_rows

print(f"Posts requested: {expected:,}")
print(f"Posts found:     {actual:,}")
print(f"Difference:      {expected - actual:,}")

Posts requested: 2,845,217
Posts found:     2,845,217
Difference:      0


---
## 3. Build one posting history per user

The filtered posting-history table still contains one row per post.

For the later stance-detection experiments, we want one row per user, with all
of that user's posts stored together as a `PostingHistory`.

Each post retains:

- `PostId`
- `PostTime`
- `Content`

The resulting table therefore has the structure:

| UserId | PostingHistory |
|---|---|
| 9 | [{PostId, PostTime, Content}, ...] |

Posts are ordered by timestamp, with the newest posts first.

In [ ]:
input_path = "../data/preprocessed/filtered_post_history.parquet"
output_path = "../data/preprocessed/user_posting_histories.parquet"

# Load the already filtered dataset
# At this point we are working with only ~2.85M posts instead of the full 18.4M
filtered_history = pq.read_table(input_path)

print(f"Posts: {filtered_history.num_rows:,}")
print(f"Columns: {filtered_history.column_names}")

display(filtered_history.slice(0, 5).to_pandas())

Posts: 2,845,217
Columns: ['UserId', 'PostId', 'PostTime', 'Content']


,UserId,PostId,PostTime,Content
0,181404,8420847,2024-11-19T05:05:01.485Z,My head hurts thinking about how illogical thi...
1,78674,3651245,2024-11-19T05:04:59.651Z,Howard Lutnick and Scott Bessent’s battle to b...
2,255778,11675952,2024-11-19T05:04:59.247Z,"Geordi: ""I always thought I could work with an..."
3,350049,16002791,2024-11-19T05:04:52.964Z,School vouchers have always been about allowin...
4,251056,11357083,2024-11-19T05:04:52.817Z,You just know trolls will get off on being ins...


In [32]:
# Sort by user and timestamp, which ensures that all posts belonging to one user are next to each other
# and that each user's posting history is ordered newest -> oldest

sorted_history = filtered_history.sort_by([
    ("UserId", "ascending"),
    ("PostTime", "descending")
])


# Find where one user's posts end and the next user's posts begin

user_ids = sorted_history["UserId"].combine_chunks()
user_ids_np = user_ids.to_numpy()

change_points = np.flatnonzero(
    user_ids_np[1:] != user_ids_np[:-1]
) + 1

# Offsets define the start/end positions of each user's posting history
offsets_np = np.concatenate([
    [0],
    change_points,
    [len(user_ids_np)]
])

offsets = pa.array(offsets_np, type=pa.int64())



posts = pa.StructArray.from_arrays(
    [
        sorted_history["PostId"].combine_chunks(),
        sorted_history["PostTime"].combine_chunks(),
        sorted_history["Content"].combine_chunks()
    ],
    names=[
        "PostId",
        "PostTime",
        "Content"
    ]
)


# Group those post structs into one list per user
posting_histories = pa.LargeListArray.from_arrays(
    offsets,
    posts
)


# Select each UserId once
unique_user_ids = pc.take(
    user_ids,
    pa.array(offsets_np[:-1], type=pa.int64())
)


# Final table: one row per user
user_histories = pa.table({
    "UserId": unique_user_ids,
    "PostingHistory": posting_histories
})

print(f"Users: {user_histories.num_rows:,}")
print(user_histories.schema)

Users: 8,467
UserId: int64
PostingHistory: large_list<item: struct<PostId: int64, PostTime: string, Content: string>>
  child 0, item: struct<PostId: int64, PostTime: string, Content: string>
      child 0, PostId: int64
      child 1, PostTime: string
      child 2, Content: string


In [ ]:
# Save the grouped posting histories
pq.write_table(
    user_histories,
    output_path,
    compression="snappy"
)

print(f"Saved to: {output_path}")


# Sanity checkes

history_lengths = (
    user_histories["PostingHistory"]
    .combine_chunks()
    .value_lengths()
)

print(f"\nUsers: {user_histories.num_rows:,}")
print(f"Total posts: {pc.sum(history_lengths).as_py():,}")
print(f"Min posts per user: {pc.min(history_lengths).as_py():,}")
print(f"Max posts per user: {pc.max(history_lengths).as_py():,}")
print(f"Average posts per user: {pc.mean(history_lengths).as_py():.1f}")


# Inspect one user and the first few posts in their history.
sample = user_histories.slice(0, 1).to_pylist()[0]

print(f"\nSample UserId: {sample['UserId']}")
print(f"Number of posts: {len(sample['PostingHistory'])}")

for post in sample["PostingHistory"][:3]:
    print("\n---")
    print("PostId:", post["PostId"])
    print("PostTime:", post["PostTime"])
    print("Content:", post["Content"][:300] if post["Content"] else None)

Saved to: ../data/preprocessed/user_posting_histories.parquet

Users: 8,467
Total posts: 2,845,217
Min posts per user: 1
Max posts per user: 1,000
Average posts per user: 336.0

Sample UserId: 9
Number of posts: 1000

---
PostId: 4686267
PostTime: 2024-11-20T18:53:26.079Z
Content: Now seems like a good time to remind people that Senators Klobuchar & Lujan proposed a law to make "health misinfo" illegal. And it would be misinfo as determined by *the head of HHS*. And now RFK Jr. is going to lead that agency.

Stop pushing for censorship
 https://anonymous

---
PostId: 12167384
PostTime: 2024-11-19T16:56:42.908Z
Content: I asked for an image of Moo Deng supporting the Rule of Law. This is what I got.

---
PostId: 3924058
PostTime: 2024-11-16T23:24:58.922Z
Content: Take it from an already busy family law lawyer … if you have a secret cell phone, turn it off on November 20th. https://anonymous


---
## 4. Join posting histories with LLM stance labels

We now combine the user posting histories with the stance annotations from
`llm_annotated_full_user_stance_dataset.parquet`.

The LLM annotation table contains one row per **user–target pair**, since the same
user can have a different stance toward Trump and Harris.

We therefore join the tables on `UserId`:

| UserId | PostingHistory |
|---|---|
| 9 | [post1, post2, ...] |

plus

| UserId | TargetEntity | StanceLabel |
|---|---|---|
| 9 | Trump | Against |
| 9 | Harris | Favor |

→

| UserId | TargetEntity | PostingHistory | StanceLabel |
|---|---|---|---|
| 9 | Trump | [post1, post2, ...] | Against |
| 9 | Harris | [post1, post2, ...] | Favor |

The posting history is therefore intentionally repeated for different target entities.

The human-annotated validation users will be handled separately in the next step so
that they can be reserved for evaluation and are not accidentally used for training.

In [22]:
llm = pq.read_table(
    "../data/raw/llm_annotated_full_user_stance_dataset.parquet"
)

# Check number of unique users
print(
    "LLM users:",
    pc.count_distinct(llm["UserId"]).as_py()
)

# Check which targets and stance labels occur
print("\nTargets:")
print(llm["TargetEntity"].value_counts())

print("\nStance labels:")
print(llm["StanceLabel"].value_counts())

LLM users: 8022

Targets:
-- is_valid: all not null
-- child 0 type: string
  [
    "Harris",
    "Trump"
  ]
-- child 1 type: int64
  [
    8022,
    8022
  ]

Stance labels:
-- is_valid: all not null
-- child 0 type: string
  [
    "Favor",
    "Against",
    "Neither"
  ]
-- child 1 type: int64
  [
    3071,
    7832,
    5141
  ]


In [ ]:
histories_path = "../data/preprocessed/user_posting_histories.parquet"
llm_labels_path = "../data/raw/llm_annotated_full_user_stance_dataset.parquet"

output_path = "../data/preprocessed/llm_user_target_histories.parquet"


# Load the posting histories 
user_histories = pq.read_table(histories_path)


# Only load the LLM annotation columns that are relevant for our dataset
# ConfidenceLevel is retained for later analysis, but will not be used as a model input
llm_labels = pq.read_table(
    llm_labels_path,
    columns=[
        "UserId",
        "TargetEntity",
        "StanceLabel",
        "ConfidenceLevel"
    ]
)

print(f"Posting-history users: {user_histories.num_rows:,}")
print(f"LLM user-target pairs: {llm_labels.num_rows:,}")
print(
    f"Unique LLM users: "
    f"{pc.count_distinct(llm_labels['UserId']).as_py():,}"
)

display(llm_labels.slice(0, 10).to_pandas())

Posting-history users: 8,467
LLM user-target pairs: 16,044
Unique LLM users: 8,022


,UserId,TargetEntity,StanceLabel,ConfidenceLevel
0,9,Harris,Favor,0.85
1,9,Trump,Against,0.95
2,10,Harris,Favor,0.95
3,10,Trump,Against,0.95
4,49,Harris,Against,0.85
5,49,Trump,Against,0.95
6,105,Harris,Neither,NaN
7,105,Trump,Against,0.95
8,144,Harris,Against,0.85
9,144,Trump,Against,0.95


In [ ]:
# Match each LLM-labelled user-target pair to its posting history
# We find the row index of each LLM UserId in user_histories and use those indices to retrieve the corresponding PostingHistory.

history_user_ids = user_histories["UserId"].combine_chunks()
posting_histories = user_histories["PostingHistory"].combine_chunks()

# For each UserId in the LLM table, find its position in user_histories
history_indices = pc.index_in(
    llm_labels["UserId"].combine_chunks(),
    value_set=history_user_ids
)

# Check whether any LLM-labelled users have no posting history
missing_histories = history_indices.null_count

print(f"LLM user-target pairs: {llm_labels.num_rows:,}")
print(f"Missing posting histories: {missing_histories:,}")

# Retrieve the corresponding posting history for every user-target pair
matched_histories = pc.take(
    posting_histories,
    history_indices
)

# Construct the final table directly
model_data = pa.table({
    "UserId": llm_labels["UserId"],
    "TargetEntity": llm_labels["TargetEntity"],
    "PostingHistory": matched_histories,
    "StanceLabel": llm_labels["StanceLabel"],
    "ConfidenceLevel": llm_labels["ConfidenceLevel"]
})

print(f"Rows after matching: {model_data.num_rows:,}")

# Save the final LLM-labelled modelling table
output_path = "../data/preprocessed/llm_user_target_histories.parquet"

pq.write_table(
    model_data,
    output_path,
    compression="snappy"
)

print(f"Saved final table to: {output_path}")

LLM user-target pairs: 16,044
Missing posting histories: 0
Rows after matching: 16,044
Saved final table to: ../data/preprocessed/llm_user_target_histories.parquet


In [27]:
# Sanity Checks

# Reload the saved file to verify that it was written correctly
final_table = pq.read_table(
    "../data/preprocessed/llm_user_target_histories.parquet"
)

print(f"Rows: {final_table.num_rows:,}")
print(f"Columns: {final_table.num_columns}")

print("\nSchema:")
print(final_table.schema)

Rows: 16,044
Columns: 5

Schema:
UserId: int64
TargetEntity: string
PostingHistory: large_list<element: struct<PostId: int64, PostTime: string, Content: string>>
  child 0, element: struct<PostId: int64, PostTime: string, Content: string>
      child 0, PostId: int64
      child 1, PostTime: string
      child 2, Content: string
StanceLabel: string
ConfidenceLevel: double


---
## 5. Build the human gold evaluation dataset

The human validation dataset contains stance annotations for 445 users.

Unlike the LLM-labelled dataset, the labels are stored in wide format:

| UserId | Trump | Harris |
|---|---|---|
| ... | Against | Favor |

For our modelling pipeline, we convert this into the same user-target format used
by the LLM-labelled dataset:

| UserId | TargetEntity | StanceLabel |
|---|---|---|
| ... | Trump | Against |
| ... | Harris | Favor |

We then attach each user's previously constructed `PostingHistory`.

The resulting human-labelled dataset will later be used as the final test set.

In [45]:
human_path = "../data/raw/human_annotated_validation_user_stance_dataset.parquet"

# The human dataset is very small (445 rows), so using pandas for reshaping is convenient and does not create any memory issues
human_wide = pq.read_table(human_path).to_pandas()

print(f"Human-annotated users: {len(human_wide):,}")

display(human_wide.head())

# Convert Trump/Harris columns into one row per user-target pair.
human_long = human_wide.melt(
    id_vars="UserId",
    value_vars=["Trump", "Harris"],
    var_name="TargetEntity",
    value_name="StanceLabel"
)

# Remove rows without a human stance annotation, if there are any.
human_long = human_long.dropna(subset=["StanceLabel"]).reset_index(drop=True)

print(f"Human user-target pairs: {len(human_long):,}")

display(human_long.head(10))
print(human_long["TargetEntity"].value_counts())

Human-annotated users: 445


,UserId,Trump,Harris
0,186791,Against,Favor
1,88089,Against,Against
2,254114,Against,Favor
3,77504,Against,Favor
4,132412,Against,Favor


Human user-target pairs: 890


,UserId,TargetEntity,StanceLabel
0,186791,Trump,Against
1,88089,Trump,Against
2,254114,Trump,Against
3,77504,Trump,Against
4,132412,Trump,Against
5,7049,Trump,Against
6,111658,Trump,Against
7,12476,Trump,Against
8,182701,Trump,Against
9,239303,Trump,Against


TargetEntity
Trump     445
Harris    445
Name: count, dtype: int64


In [ ]:
user_histories = pq.read_table(
    "../data/preprocessed/user_posting_histories.parquet"
)

# Convert the small human metadata table back to Arrow
human_labels = pa.Table.from_pandas(
    human_long,
    preserve_index=False
)

history_user_ids = user_histories["UserId"].combine_chunks()
posting_histories = user_histories["PostingHistory"].combine_chunks()

# Find the posting-history row corresponding to every human-labelled user
history_indices = pc.index_in(
    human_labels["UserId"].combine_chunks(),
    value_set=history_user_ids
)

print(
    f"Human user-target pairs: {human_labels.num_rows:,}"
)
print(
    f"Missing posting histories: {history_indices.null_count:,}"
)

# Retrieve the appropriate posting history
matched_histories = pc.take(
    posting_histories,
    history_indices
)

# Construct the human gold table
human_gold = pa.table({
    "UserId": human_labels["UserId"],
    "TargetEntity": human_labels["TargetEntity"],
    "PostingHistory": matched_histories,
    "StanceLabel": human_labels["StanceLabel"]
})

print(f"Rows in human gold dataset: {human_gold.num_rows:,}")

Human user-target pairs: 890
Missing posting histories: 0
Rows in human gold dataset: 890


In [ ]:
# Check for duplicate human user-target pairs
human_check = human_gold.select([
    "UserId",
    "TargetEntity",
    "StanceLabel"
]).to_pandas()

duplicates = human_check.duplicated(
    subset=["UserId", "TargetEntity"]
).sum()

print(f"Duplicate user-target pairs: {duplicates:,}")
print(
    f"Unique human users: "
    f"{pc.count_distinct(human_gold['UserId']).as_py():,}"
)

display(human_check.head(10))


# Save the human gold dataset
human_output_path = "../data/preprocessed/human_gold_user_target_histories.parquet"

pq.write_table(
    human_gold,
    human_output_path,
    compression="snappy"
)

print(f"\nSaved to: {human_output_path}")

Duplicate user-target pairs: 0
Unique human users: 445


,UserId,TargetEntity,StanceLabel
0,186791,Trump,Against
1,88089,Trump,Against
2,254114,Trump,Against
3,77504,Trump,Against
4,132412,Trump,Against
5,7049,Trump,Against
6,111658,Trump,Against
7,12476,Trump,Against
8,182701,Trump,Against
9,239303,Trump,Against



Saved to: ../data/preprocessed/human_gold_user_target_histories.parquet


---
## 6. Reserve all human-annotated users for final evaluation

The human-annotated users form our final gold test set.

Some of these users may also occur in the LLM-labelled dataset. To avoid data
leakage, we remove **all rows belonging to human-annotated users** from the
LLM-labelled training pool.

In [39]:
llm_data = pq.read_table(
    "../data/preprocessed/llm_user_target_histories.parquet"
)

human_user_ids = human_gold["UserId"].combine_chunks().unique()
llm_user_ids = llm_data["UserId"].combine_chunks().unique()

# Identify which human users also occur in the LLM-labelled dataset.
overlap_mask = pc.is_in(
    human_user_ids,
    value_set=llm_user_ids
)

overlapping_users = human_user_ids.filter(overlap_mask)

print(f"LLM users: {len(llm_user_ids):,}")
print(f"Human users: {len(human_user_ids):,}")
print(f"Overlap: {len(overlapping_users):,} users")

LLM users: 8,022
Human users: 445
Overlap: 0 users


In [47]:
# The human gold users and LLM-labelled users are completely disjoint, so no users need to be removed from the LLM training pool
assert len(overlapping_users) == 0, "Unexpected overlap between LLM and human users."

llm_pool = llm_data

print(f"LLM user-target pairs available for train/validation: {llm_pool.num_rows:,}")
print(f"LLM users available for train/validation: {len(llm_user_ids):,}")

LLM user-target pairs available for train/validation: 16,044
LLM users available for train/validation: 8,022


---
## 7. Create train and validation splits

The remaining LLM-labelled users are divided into training and validation sets.

The split is performed at the **user level** rather than the user-target level.
This ensures that all target examples belonging to the same user remain in the
same split and prevents the same posting history from appearing in both training
and validation.

We use:

- 80% of the remaining LLM users for training
- 20% for validation
- all human-annotated users as the final test set

A fixed random seed is used to make the split reproducible.

In [41]:
from sklearn.model_selection import train_test_split

RANDOM_SEED = 42

# Extract each remaining LLM user exactly once.
remaining_users = (
    llm_pool["UserId"]
    .combine_chunks()
    .unique()
    .to_numpy()
)

train_users, validation_users = train_test_split(
    remaining_users,
    test_size=0.20,
    random_state=RANDOM_SEED,
    shuffle=True
)

print(f"Training users:   {len(train_users):,}")
print(f"Validation users: {len(validation_users):,}")
print(f"Human test users: {len(human_user_ids):,}")

Training users:   6,417
Validation users: 1,605
Human test users: 445


In [42]:
train_user_ids = pa.array(train_users, type=pa.int64())
validation_user_ids = pa.array(validation_users, type=pa.int64())


# Select all target examples belonging to training users.
train_mask = pc.is_in(
    llm_pool["UserId"].combine_chunks(),
    value_set=train_user_ids
)

train_data = llm_pool.filter(train_mask)


# Select all target examples belonging to validation users.
validation_mask = pc.is_in(
    llm_pool["UserId"].combine_chunks(),
    value_set=validation_user_ids
)

validation_data = llm_pool.filter(validation_mask)


print(f"Training user-target pairs:   {train_data.num_rows:,}")
print(f"Validation user-target pairs: {validation_data.num_rows:,}")
print(f"Human test user-target pairs: {human_gold.num_rows:,}")

Training user-target pairs:   12,834
Validation user-target pairs: 3,210
Human test user-target pairs: 890


In [43]:
train_ids = set(
    train_data["UserId"].combine_chunks().unique().to_pylist()
)

validation_ids = set(
    validation_data["UserId"].combine_chunks().unique().to_pylist()
)

test_ids = set(
    human_gold["UserId"].combine_chunks().unique().to_pylist()
)


print("Train ∩ validation:", len(train_ids & validation_ids))
print("Train ∩ test:", len(train_ids & test_ids))
print("Validation ∩ test:", len(validation_ids & test_ids))

Train ∩ validation: 0
Train ∩ test: 0
Validation ∩ test: 0


---
## 8. Save the final dataset splits

The preprocessing pipeline produces three independent datasets:

- `train.parquet`: LLM-labelled training examples
- `validation.parquet`: LLM-labelled examples used during model development
- `human_test.parquet`: human-labelled gold examples used only for final evaluation

The human test set must not be used for model training, model selection, or
hyperparameter tuning.

In [44]:
train_path = "../data/preprocessed/train.parquet"
validation_path = "../data/preprocessed/validation.parquet"
test_path = "../data/preprocessed/human_test.parquet"

pq.write_table(
    train_data,
    train_path,
    compression="snappy"
)

pq.write_table(
    validation_data,
    validation_path,
    compression="snappy"
)

pq.write_table(
    human_gold,
    test_path,
    compression="snappy"
)

print("Saved:")
print(f"  Train:      {train_path}")
print(f"  Validation: {validation_path}")
print(f"  Human test: {test_path}")

Saved:
  Train:      ../data/preprocessed/train.parquet
  Validation: ../data/preprocessed/validation.parquet
  Human test: ../data/preprocessed/human_test.parquet
